In [1]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
from ITIS_Model_funcs import get_nominal_param,OLS_res,ITIS

In [2]:
param_log,IC = get_nominal_param()

pickles = ['11', '12', '21', '22', '31', '32']

with open('OLS_Results\\ResAnalysis11.pkl', 'rb') as f:
    results = pickle.load(f)

        ##Reference
        # all_results = {
        #     'output_ids': output_ids,
        #     'param_ids' : param_ids,
        #     'param_in' : param_in,
        #     'twonorms' : twonorms,
        #     't_data' : t_data,
        #     'y_data' : y_data,
        #     'opt_model' : opt_model,
        #     'true_sol' : true_sol,
        #     'param_opt' : param_opt,
        #     'optimal_solution' :np.exp(least_sq_sol.x),
        #     'objvalue' : least_sq_sol.cost
        # }


output_ids = results['output_ids']
t_data = results['t_data']
param_opt = results['param_opt']
y_data = results['y_data']
param_ids = results['param_ids']

##Sensitvity analysis of residuals Starting from optimized param values?

##SENSITIVITY ANALYSIS
h = 1e-6  #amount to perturb parameters
n_param = len(param_opt)
n_states = len(output_ids)

S = np.zeros((n_param, len(t_data) * n_states))  ##Initialize shape of sensitivity matrix.


##Reference
# for i in range(n_param):  #calculate the relative sensitivity to each 45 parameters
#
#         param_in = param_log[i]
#         param_delta = param_in + h ##    [HOW EXACTLY SHOULD I BE PERTURBING MY PARAM?]
#         ##THIS THING NEXT DO THIS!!!
#         Sensitivity_Mat[i, :] = ((1 / h) * (OLS_res(param_delta, y_data, t_data, i, output_ids, param_all, IC)
#                                             - OLS_res(param_in, y_data, t_data, i, output_ids, param_all, IC)))
for i in range(n_param):  #calculate the relative sensitivity to each 45 parameters

        param_in = param_opt[i]
        param_delta = param_in + h ##    [HOW EXACTLY SHOULD I BE PERTURBING MY PARAM?]
        ##THIS THING NEXT DO THIS!!!
        S[i, :] = ((1 / h) * (OLS_res(param_delta, y_data, t_data, i, output_ids, param_log, IC)
                                            - OLS_res(param_in, y_data, t_data, i, output_ids, param_log, IC)))

In [3]:
F = S@S.T
print(np.shape(F))
np.linalg.cond(F)

(45, 45)


np.float64(9867478140700.469)

In [4]:
S_opt = S[param_ids,:]
F_opt = S_opt@S_opt.T
np.linalg.cond(F_opt)
C = np.linalg.inv(F_opt)
print(np.diag(C))
print(np.exp(param_opt[param_ids]))
print(np.diag(C)/np.exp(param_opt[param_ids]))


[0.00240548 0.00241232 0.00521131 0.09304717 0.01302659 0.04705096
 0.07391414 0.0236107  0.32965944 0.00245302]
[1.51808303e-02 2.01028096e-01 3.04510970e-02 8.74918731e+01
 2.95522722e-09 5.00624103e-04 8.43646030e+01 1.89949090e+03
 1.14852604e-01 8.50500412e+02]
[1.58455133e-01 1.19999335e-02 1.71137028e-01 1.06349502e-03
 4.40798332e+06 9.39846135e+01 8.76127454e-04 1.24300167e-05
 2.87028268e+00 2.88420651e-06]


In [28]:
##import rankings
with open('paramRankings\\rankingDesign1.pkl', 'rb') as f:
    results = pickle.load(f)

rank_value = results['rank_value']
param_sorted = results['param_sorted']

param_titles = ['d1',
'k1','k2','h1','h2','h3','d2',
'k3','k4','h4','d3',
'h5','h6','k5','k6','h7','d4',
'b1','k7','h8','k8','h9','d5','h10',
'b2','k9','k10','k11','d6',
'k12','k13','k14','h11','d7',
'k15','k16','d8',
'alpha', 'k', 'beta', 'L', 'eps', 'delta', 'T', 'Nc']

circadian_param = ['alpha', 'k', 'beta', 'L', 'eps', 'delta', 'T', 'Nc']

cond = 0
select = 10
while cond < 1e+5:
    select +=1
    p_indices = [param_titles.index(param_sorted[i]) for i in range(select)]
    S_opt = S[p_indices,:]
    F_opt = S_opt@S_opt.T
    cond = np.linalg.cond(F_opt)

select -= 1
p_indices = [param_titles.index(param_sorted[i]) for i in range(select)]
S_opt = S[p_indices,:]
F_opt = S_opt@S_opt.T
C_opt = np.linalg.inv(F_opt)
print("Number of selected parameters:", select)
print("Selected Parameters: ", [param_titles[i] for i in p_indices] )
print(np.linalg.cond(F_opt))
print(np.diag(C_opt))
print(np.exp(param_opt[param_ids]))
print(np.diag(C_opt)/np.exp(param_opt[p_indices]))

Number of selected parameters: 25
Selected Parameters:  ['d7', 'h6', 'd8', 'h11', 'T', 'beta', 'd4', 'k3', 'k14', 'k4', 'h7', 'alpha', 'h4', 'd6', 'k15', 'k6', 'k1', 'k10', 'd1', 'd3', 'k16', 'h2', 'Nc', 'k9', 'L']
94916.1968608437
[0.03843517 0.09580346 0.13421462 1.04852885 0.04396234 0.09350517
 1.62660513 0.01997153 0.14326548 9.74999274 0.22796196 0.35708842
 0.01905758 0.67226329 0.22367916 0.09008602 0.10651517 2.48634245
 0.09262558 0.06408602 0.0987929  0.05253151 0.41935776 0.59775265
 0.1745296 ]
[1.51808303e-02 2.01028096e-01 3.04510970e-02 8.74918731e+01
 2.95522722e-09 5.00624103e-04 8.43646030e+01 1.89949090e+03
 1.14852604e-01 8.50500412e+02]
[2.53182290e+00 4.76567537e-01 4.40754635e+00 1.19842999e-02
 3.05293997e-05 9.84264913e-05 5.55155334e+01 6.75803531e+06
 1.69817050e-03 8.48913509e+01 1.20012135e-04 1.19029473e-03
 2.24074941e-05 2.10082277e+01 4.46800627e+02 2.53663391e-06
 2.13217971e-09 1.41607384e-09 6.86115400e+05 2.01674217e+00
 8.23274163e-03 7.28390316e-